In [1]:
import subprocess
import sys
import json
import os
import time
import numpy as np
import math
import random
from scipy import stats
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd
from seaborn import violinplot

# Генерируем временные ряды

In [2]:
df = pd.DataFrame(columns=["filename", "N", "distrib_1", "args_1", "distrib_2", "args_2", "change_point", "cp_type"])

In [3]:
with open('config.json', 'r', encoding='utf-8') as f:
    config = json.load(f)

distribs = config['distribs']
Ns = config['Ns']
reps = config['reps']
seed = config['seed']
folder = "numbers"

Разладки бывают:
- носитель не меняется
    - в среднем (std = const) (правда ли что сложность зависит от $(\mu_1 - \mu_2)/std$) (1)
        - norm
        - lap
        - gam (осторожно с параметрами)
    - в дисперсии (mean = const) (2)
        - norm
        - lap
        - gam (осторожно с параметрами)
    - в то, и то (3)
        - norm
        - lap
        - exp
        - gam (осторожно с параметрами)
    - в кучности (эксцесс) (4)
        - norm $\longleftrightarrow$ lap $\longleftrightarrow$ uni
    - в скосе (skew)
        - gamma $\longleftrightarrow$ exp
        - gamma $\longleftrightarrow$ gamma
- экзотические случаи (меняется носитель или форма)
    - носитель сдвигается, но ограничен (5)
        - uni $\longleftrightarrow$ uni
        - exp $\longleftrightarrow$ exp (shift)
    - распределение симметричное, носитель становится/перестает быть ограниченным (6)
        - uni $\longleftrightarrow$ norm
    - смена формы (7)
        - norm $\longleftrightarrow$ 2norm
    - появление примеси
        - norm $\longleftrightarrow$ (norm + exp)/2

In [4]:
for distribs_ in distribs:
    for N_ in Ns:
        for _ in range(reps):
            output_file = f"numbers_{len(df)}.txt"
            change_point = random.randint(N_ // 4, N_ * 3 // 4)
            seed = random.randint(0, 100)
            first_distrib_ = distribs_[0]
            input_args = list(map(str, [change_point, first_distrib_[0], output_file, seed, *first_distrib_[1:]]))
            command = ["./setgen"] + input_args
            process = subprocess.run(
                command,
                text=True,
                stdout=subprocess.PIPE
            )
            with open(folder + '/' + output_file, 'r', encoding='utf-8') as f:
                file = f.read()
            
            second_distrib_ = distribs_[1]
            input_args = list(map(str, [N_ - change_point, second_distrib_[0], output_file, seed, *second_distrib_[1:]]))
            command = ["./setgen"] + input_args
            process = subprocess.run(
                command,
                text=True,
                stdout=subprocess.PIPE
            )
            with open(folder + '/' + output_file, 'r', encoding='utf-8') as f:
                file += f.read()  
            with open(folder + '/' + output_file, 'w', encoding='utf-8') as f:
                f.write(file) 
            
            df.loc[len(df)] = {"filename": output_file, "N": N_, "distrib_1": first_distrib_[0], \
                               "args_1": first_distrib_[1:], "distrib_2": second_distrib_[0], \
                               "args_2": second_distrib_[1:], "change_point": change_point, "cp_type": distribs_[2]}

In [5]:
df

,filename,N,distrib_1,args_1,distrib_2,args_2,change_point,cp_type
0,numbers_0.txt,100,norm,"[0, 1]",norm,"[5, 1]",45,1
1,numbers_1.txt,100,norm,"[0, 1]",norm,"[5, 1]",36,1
2,numbers_2.txt,100,norm,"[0, 1]",norm,"[5, 1]",60,1
3,numbers_3.txt,100,norm,"[0, 1]",norm,"[5, 1]",43,1
4,numbers_4.txt,100,norm,"[0, 1]",norm,"[5, 1]",64,1
...,...,...,...,...,...,...,...,...
515,numbers_515.txt,1000,2norm,"[1, 1]",norm,"[0, 2]",668,7
516,numbers_516.txt,1000,2norm,"[1, 1]",norm,"[0, 2]",636,7
517,numbers_517.txt,1000,2norm,"[1, 1]",norm,"[0, 2]",469,7
518,numbers_518.txt,1000,2norm,"[1, 1]",norm,"[0, 2]",561,7


# А теперь запускаем алгоритмы

In [451]:
checkers = ["./concomp_online", "./edges_online", "./triang_online"]
Ws = [lambda x: 3, lambda x: x//10, lambda x: math.sqrt(x), lambda x: x]
deltas = [lambda x: 0.1, lambda x: 1/math.log(x), lambda x: 1/math.sqrt(x), lambda x: 1/x]
deltas_names = ["const", "1/logW", "1/sqrtW", "1/W"]

In [452]:
cp_df = pd.DataFrame(columns=["filename", "checker", "N", "window", "delta", "predict", "change_point", "hit", "miss"])

In [453]:
def hit_calc(result: list, cp: int, N: int):
    hit_rate = 0
    for i in result:
        if i > cp:
            hit_rate += 1/(i - cp)
    return hit_rate
def miss_calc(result: list, cp: int, N: int):
    miss_rate = 0
    for i in result:
        if i < cp:
            miss_rate += 1
    return miss_rate

In [454]:
reps = len(df) * len(checkers) * len(Ws) * len(deltas)

with tqdm(total=reps, unit="%") as pbar:
    rep = 0
    for checker in checkers:
        for func in Ws:
            for did in range(len(deltas)):
                for index in range(len(df)):
                    rep += 1
                    pbar.update(rep - pbar.n)
                    
                    file = df.iloc[index, 0]
                    N_ = round(func(df.iloc[index, 1]))
                    delta_ = deltas[did](N_)
                    
                    input_args = list(map(str, [N_, delta_, file]))
                    command = [checker] + input_args
                    process = subprocess.run(
                        command,
                        text=True,
                        stdout=subprocess.PIPE
                    )
                    result = list(map(int, process.stdout[:-2].split()))
                
                    cp = df.iloc[index, 6]
                    Size = df.iloc[index, 1]
                    cp_df.loc[len(cp_df)] = {"filename": file, "checker": checker[2:-7], "N": Size, \
                                             "window": N_, "delta": deltas_names[did], "predict": result, "change_point": cp, \
                                             "hit": hit_calc(result, cp, Size), "miss": miss_calc(result, cp, Size)}

100%|██████████████████████████████████████| 24960/24960 [07:09<00:00, 58.13%/s]


# А теперь Database analysis

In [475]:
cp_df.tail(5)

,filename,checker,N,window,delta,predict,change_point,hit,miss
24955,numbers_515.txt,triang,1000,1000,1/W,"[390, 391, 392, 393, 394, 395, 396, 397, 398, ...",578,0.903009,65
24956,numbers_516.txt,triang,1000,1000,1/W,"[579, 580, 581, 582, 583, 584, 585, 586, 587, ...",411,0.933402,0
24957,numbers_517.txt,triang,1000,1000,1/W,"[365, 366, 367, 368, 369, 370, 371, 372, 373, ...",558,2.629443,89
24958,numbers_518.txt,triang,1000,1000,1/W,"[290, 291, 292, 293, 294, 295, 296, 297, 298, ...",335,0.555308,32
24959,numbers_519.txt,triang,1000,1000,1/W,"[389, 390, 391, 392, 393, 394, 395, 396, 397, ...",358,1.319560,0


In [476]:
cp_df.sort_values("hit", ascending=False).head(5)

,filename,checker,N,window,delta,predict,change_point,hit,miss
24750,numbers_310.txt,triang,1000,1000,1/W,"[386, 387, 388, 389, 390, 391, 392, 393, 394, ...",571,6.461780,79
24592,numbers_152.txt,triang,1000,1000,1/W,"[295, 296, 297, 298, 299, 300, 301, 302, 303, ...",645,6.450741,78
24599,numbers_159.txt,triang,1000,1000,1/W,"[380, 381, 382, 383, 384, 385, 386, 387, 388, ...",653,6.397077,112
24754,numbers_314.txt,triang,1000,1000,1/W,"[403, 404, 405, 406, 407, 408, 409, 410, 411, ...",429,6.280162,26
24597,numbers_157.txt,triang,1000,1000,1/W,"[451, 452, 453, 454, 455, 456, 457, 458, 459, ...",717,6.137840,166


In [477]:
cp_df.merge(df.drop(["N", "change_point"], axis=1), on="filename").drop(["predict", "change_point"], axis=1) \
    .query("miss < 10") \
    .sort_values("hit", ascending=False).head(5)

,filename,checker,N,window,delta,hit,miss,distrib_1,args_1,distrib_2,args_2,cp_type
24895,numbers_455.txt,triang,1000,1000,1/W,5.951886,5,uni,"[-1.73, 1.73]",norm,"[0, 1]",6
24611,numbers_171.txt,triang,1000,1000,1/W,4.586772,8,norm,"[0, 3]",norm,"[3, 1]",3
24798,numbers_358.txt,triang,1000,1000,1/W,4.569248,6,norm,"[0, 1]",uni,"[-1.2, 1.2]",4
5018,numbers_338.txt,concomp,1000,32,1/logW,4.447080,0,uni,"[-1.2, 1.2]",norm,"[0, 1]",4
3015,numbers_415.txt,concomp,1000,100,1/logW,4.437282,0,uni,"[0, 1]",uni,"[2, 3]",5


In [478]:
cp_df.groupby(["checker"]).agg({"hit": "mean", "miss" : "mean"})

,hit,miss
checker,,
concomp,0.178970,0.414663
edges,0.103362,1.910937
triang,0.307342,8.788582


In [479]:
df.head(2)

,filename,N,distrib_1,args_1,distrib_2,args_2,change_point,cp_type
0,numbers_0.txt,100,norm,"[0, 1]",norm,"[5, 1]",51,1
1,numbers_1.txt,100,norm,"[0, 1]",norm,"[5, 1]",59,1


In [480]:
cp_df.head(2)

,filename,checker,N,window,delta,predict,change_point,hit,miss
0,numbers_0.txt,concomp,100,3,const,[70],51,0.052632,0
1,numbers_1.txt,concomp,100,3,const,[84],59,0.040000,0


In [481]:
tmp_df = cp_df.merge(df[["filename", "cp_type"]], on="filename").drop(["predict", "change_point"], axis=1)

In [483]:
tmp_df.head(2)

,filename,checker,N,window,delta,hit,miss,cp_type
0,numbers_0.txt,concomp,100,3,const,0.052632,0,1
1,numbers_1.txt,concomp,100,3,const,0.040000,0,1


In [485]:
tmp_df.groupby(["cp_type", "checker"]).agg({"hit": "mean", "miss" : "mean"})

hit       miss
cp_type checker                     
1       concomp  0.152584   0.218750
        edges    0.016101   1.838281
        triang   0.088476   9.622656
2       concomp  0.107808   0.404687
        edges    0.204075   2.743750
        triang   0.456205  10.164062
3       concomp  0.191908   0.618229
        edges    0.160604   2.261458
        triang   0.440529   9.448958
4       concomp  0.073270   0.393750
        edges    0.085279   1.158854
        triang   0.336655   7.530729
5       concomp  0.969983   0.451562
        edges    0.079659   1.100000
        triang   0.127612   5.057812
6       concomp  0.018873   0.175000
        edges    0.027005   1.476562
        triang   0.211223   8.775000
7       concomp  0.021443   0.481250
        edges    0.059045   2.840625
        triang   0.235694   9.906250